# Aula 02 (Prática de Nivelamento): Introdução ao Machine Learning — De Regressões às Redes Neurais

Bem-vindo(a) a este notebook de **nivelamento prático** da **Aula 02**! 🚀

Como percebemos que muitos alunos vieram diretamente para o curso de Deep Learning sem uma bagagem prévia de Machine Learning clássico, este notebook foi criado sob medida para conectar as pontas e construir uma base sólida antes de avançarmos para Autograd e Redes Neurais Multicamadas (MLP).

---

### 💡 A Filosofia Central e o Uso de Widgets Interativos
Em desenvolvimento de software tradicional, nós escrevemos **regras (código)** e passamos **dados** para obter **respostas**:
$$\text{Dados} + \text{Regras} \longrightarrow \text{Respostas}$$

Em Machine Learning e Deep Learning, invertemos o processo! Passamos **dados** e as **respostas desejadas** para que o computador **encontre a regra (a equação matemática com coeficientes $w$ e $b$)**:
$$\text{Dados} + \text{Respostas} \longrightarrow \text{Equação (Coeficientes)}$$

Neste notebook, vamos desmistificar essa busca pelos **coeficientes corretos da equação** ($w$ e $b$) através de **Simuladores Interativos (`ipywidgets`)**. Você poderá arrastar os sliders para:

1. **Ajustar Manualmente os Coeficientes**: Entender o papel visual de $w$ (inclinação) e $b$ (deslocamento) e ver a norma dos erros residuais mudar.
2. **Navegar Época por Época no Gradiente Descendente**: Acompanhar passo a passo o algoritmo ajustando as curvas e observando em tempo real como a **Loss (Erro)** e a **Norma do Gradiente ($\|\nabla L\|$)** diminuem até a convergência.
3. **Conectar com Redes Neurais (MLP)**: Ver como 1 neurônio linear ou sigmoide se expande para multicamadas no PyTorch.


In [ ]:
#@title Clique aqui para ver o código { display-mode: "form" }
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider, Layout

# Fixando a semente aleatória para reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

print("🔥 Ambiente preparado com sucesso com ipywidgets!")
print(f"Versão do PyTorch: {torch.__version__}")
print(f"Versão do ipywidgets: {widgets.__version__}")


---

## 1. Regressão Linear: Ajustando uma Reta a Pontos em 2D

Em uma **Regressão Linear**, queremos prever um valor contínuo $y$ a partir de uma entrada $x$. A relação entre eles é modelada por uma equação de primeiro grau (uma reta no gráfico 2D):

$$y = w \cdot x + b$$

Onde:
* **$x$**: A variável de entrada (*feature*), ex: **Anos de Experiência em Programação**.
* **$y$**: A variável que queremos prever (*target*), ex: **Salário em R$ 1.000**.
* **$w$ (Peso / Weight / Inclinação)**: Define a inclinação da reta (quanto o salário cresce para cada ano de experiência).
* **$b$ (Viés / Bias / Intercepto)**: Define onde a reta corta o eixo Y (o salário base inicial sem experiência).

### 🎯 O Erro Quadrático Médio (MSE) e a Norma dos Resíduos:
Para medir a distância entre as previsões $\hat{y}_i$ e os pontos reais $y_i$, calculamos a Perda MSE:

$$\text{MSE} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2$$

Também calculamos a **Norma Euclidiana ($L_2$) do Vetor de Erros Residuais**:

$$\|E\|_2 = \sqrt{\sum_{i=1}^{N} (\hat{y}_i - y_i)^2} = \sqrt{N \cdot \text{MSE}}$$


In [ ]:
#@title Clique aqui para ver o código { display-mode: "form" }
# 1. Gerando dados fictícios: 25 desenvolvedores (Anos de Experiência vs Salário)
X_lin = torch.linspace(0, 10, 25).unsqueeze(1)  # Shape (25, 1)

# Regra real oculta da natureza: y = 2.5 * x + 4.0 + ruído
true_w = 2.5
true_b = 4.0
noise = torch.randn_like(X_lin) * 1.2
Y_lin = true_w * X_lin + true_b + noise

x_plot = np.linspace(0, 10, 100)

# 2. Função de atualização interativa para ajuste manual de w e b
def explorar_reta_manual(w=-1.5, b=28.0):
    y_pred = w * X_lin.numpy() + b
    residuos = y_pred - Y_lin.numpy()
    mse = np.mean(residuos**2)
    norma_residuos = np.linalg.norm(residuos)
    
    plt.figure(figsize=(9, 5))
    plt.scatter(X_lin.numpy(), Y_lin.numpy(), color='#0A345D', label='Dados Reais dos Devs', s=60, alpha=0.8)
    
    y_line = w * x_plot + b
    plt.plot(x_plot, y_line, color='#1BB5D8', linewidth=2.5, label=f'Sua Reta: y = {w:.2f}x + {b:.2f}')
    
    # Linhas de resíduo pontilhadas (erros de cada ponto)
    for i in range(len(X_lin)):
        plt.vlines(X_lin.numpy()[i], Y_lin.numpy()[i], y_pred[i], color='#FF7043', linestyle=':', alpha=0.6)
        
    plt.title(f'Ajuste Manual | MSE (Erro): {mse:.2f} | Norma do Erro ||E||: {norma_residuos:.2f}', fontsize=12)
    plt.xlabel('Anos de Experiência em Programação (X)', fontsize=11)
    plt.ylabel('Salário em R$ 1.000 (Y)', fontsize=11)
    plt.legend(fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.show()

# Widget interativo com sliders de w e b
interact(
    explorar_reta_manual,
    w=FloatSlider(min=-4.0, max=6.0, step=0.1, value=-1.5, description='Peso (w):'),
    b=FloatSlider(min=-5.0, max=35.0, step=0.5, value=28.0, description='Viés (b):')
);


### 🔄 O Método Iterativo (Gradient Descent) e a Queda da Norma do Gradiente

Como o PyTorch ajusta automaticamente $w$ e $b$?

A cada **época (iteração)**, o algoritmo executa:
1. **Forward Pass**: Calcula as previsões $\hat{y} = w \cdot x + b$.
2. **Cálculo da Loss**: Calcula o erro atual ($\text{MSE}$).
3. **Backward Pass**: O PyTorch calcula os gradientes ($\frac{\partial L}{\partial w}$ e $\frac{\partial L}{\partial b}$).
4. **Atualização dos Coeficientes**: Ajusta $w$ e $b$ na direção oposta ao gradiente.

---

### 📐 O que é a Norma do Gradiente ($\|\nabla L\|$) e por que ela é importante?

O **gradiente** nada mais é do que um vetor contendo as derivadas da perda em relação a cada parâmetro:
$$\nabla L = \left( \frac{\partial L}{\partial w}, \frac{\partial L}{\partial b} \right)$$

A **Norma do Gradiente** (norma Euclidiana $L_2$) mede o **comprimento/tamanho desse vetor de inclinação**:

$$\|\nabla L\|_2 = \sqrt{\left(\frac{\partial L}{\partial w}\right)^2 + \left(\frac{\partial L}{\partial b}\right)^2}$$

### 💡 Por que observar a Norma do Gradiente?
1. **Indicador de Inclinação do Terreno**: No início (época 0), os parâmetros estão muito errados e o terreno da perda é íngreme. Por isso, a norma é alta ($\|\nabla L\| \gg 0$).
2. **Sinalizador de Convergência**: Conforme a reta se ajusta aos dados e chega ao fundo do vale (mínimo da função de perda), o terreno fica plano e a norma do gradiente **tende a zero ($\|\nabla L\| \to 0$)**.
3. **Diagnóstico de Saúde do Treinamento**: Se a norma explodir ($\|\nabla L\| \to \infty$), a taxa de aprendizado está alta demais. Se chegar a zero rápido demais sem reduzir a perda, o modelo travou em um platô.


In [ ]:
#@title Clique aqui para ver o código { display-mode: "form" }
# 1. Executando o treinamento e gravando o histórico completo de 150 épocas
w_lin = torch.tensor([[-1.5]], requires_grad=True)
b_lin = torch.tensor([[28.0]], requires_grad=True)
optimizer = optim.Adam([w_lin, b_lin], lr=0.2)

historico_lin = []

for epoch in range(151):
    pred = X_lin @ w_lin + b_lin
    loss = torch.mean((pred - Y_lin) ** 2)
    
    optimizer.zero_grad()
    loss.backward()
    
    g_w = w_lin.grad.item()
    g_b = b_lin.grad.item()
    grad_norm = np.sqrt(g_w**2 + g_b**2)
    
    historico_lin.append({
        'epoch': epoch,
        'w': w_lin.item(),
        'b': b_lin.item(),
        'loss': loss.item(),
        'grad_w': g_w,
        'grad_b': g_b,
        'grad_norm': grad_norm
    })
    
    optimizer.step()

df_hist_lin = pd.DataFrame(historico_lin)

# 2. Visualização Interativa por Época com ipywidgets
def visualizar_epoca_lin(epoca=0):
    dados_ep = df_hist_lin.iloc[int(epoca)]
    w_curr, b_curr = dados_ep['w'], dados_ep['b']
    loss_curr, gnorm_curr = dados_ep['loss'], dados_ep['grad_norm']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Subplot 1: Reta se ajustando aos pontos no eixo 2D
    ax1.scatter(X_lin.numpy(), Y_lin.numpy(), color='#0A345D', label='Dados Reais', s=50, alpha=0.8)
    
    y_curr = w_curr * x_plot + b_curr
    ax1.plot(x_plot, y_curr, color='#7CB342', linewidth=2.5, 
             label=f'Época {int(epoca)}: y = {w_curr:.2f}x + {b_curr:.2f}')
    
    ax1.set_title(f'1. A Reta Se Ajustando aos Pontos (Época {int(epoca)})', fontsize=11)
    ax1.set_xlabel('Anos de Experiência (X)')
    ax1.set_ylabel('Salário em R$ 1.000 (Y)')
    ax1.legend(fontsize=9, loc='upper left')
    ax1.grid(True, alpha=0.3)
    
    # Subplot 2: Queda da Loss MSE e da Norma do Gradiente ||∇L||
    ax2_twin = ax2.twinx()
    
    l1 = ax2.plot(df_hist_lin['epoch'][:int(epoca)+1], df_hist_lin['loss'][:int(epoca)+1], 
                  color='#0A345D', linewidth=2, label='MSE Loss (Erro)')
    ax2.scatter(int(epoca), loss_curr, color='#FF7043', s=70, zorder=5)
    
    l2 = ax2_twin.plot(df_hist_lin['epoch'][:int(epoca)+1], df_hist_lin['grad_norm'][:int(epoca)+1], 
                       color='#7CB342', linestyle='--', linewidth=2, label='Norma do Gradiente ||∇L||')
    ax2_twin.scatter(int(epoca), gnorm_curr, color='#1BB5D8', s=70, zorder=5)
    
    ax2.set_xlim(0, 150)
    ax2.set_ylim(0, df_hist_lin['loss'].max() * 1.05)
    ax2_twin.set_ylim(0, df_hist_lin['grad_norm'].max() * 1.05)
    
    ax2.set_title('2. Queda da Perda (MSE) e Diminuição da Norma do Gradiente', fontsize=11)
    ax2.set_xlabel('Época (Iteração)')
    ax2.set_ylabel('MSE Loss', color='#0A345D')
    ax2_twin.set_ylabel('Norma do Gradiente ||∇L||', color='#7CB342')
    
    lines = l1 + l2
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 [ÉPOCA {int(epoca):03d}]  w = {w_curr:6.3f} | b = {b_curr:6.3f} | MSE Loss = {loss_curr:6.3f} | Norma do Gradiente ||∇L|| = {gnorm_curr:6.3f}")

slider_ep_lin = IntSlider(min=0, max=150, step=1, value=0, description='Época:', layout=Layout(width='65%'))
interact(visualizar_epoca_lin, epoca=slider_ep_lin);


---

## 2. Regressão Logística: Classificação e a Curva Sigmoide

Agora imagine um problema diferente: em vez de prever um número contínuo (salário), queremos prever se um desenvolvedor **foi aprovado ou não em uma entrevista técnica** (0 = Não Aprovado, 1 = Aprovado) com base nas suas **Horas de Estudo Semanal**.

### Por que a Reta Simples Falha na Classificação?
Uma reta $z = w \cdot x + b$ produz valores que vão de $-\infty$ a $+\infty$. No entanto, para classificar, precisamos de uma **probabilidade entre $0.0$ e $1.0$**.

### 💡 A Solução: A Função Sigmoide $\sigma(z)$
Passamos o resultado linear $z = w \cdot x + b$ por uma função especial chamada **Sigmoide**:

$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-(w \cdot x + b)}}$$

A Sigmoide esmaga qualquer valor para o intervalo $[0, 1]$:
* Se $z = 0 \implies \sigma(0) = 0.5$ (**Fronteira de Decisão** / Divisor de Águas!).
* A **Fronteira de Decisão** no eixo X ocorre onde $w \cdot x + b = 0 \implies x = -\frac{b}{w}$.

### 🎯 A Função de Perda BCE (Binary Cross-Entropy):
$$\text{BCE} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$


In [ ]:
#@title Clique aqui para ver o código { display-mode: "form" }
# 1. Gerando dados fictícios: 30 devs (Horas de Estudo vs Aprovado [0 ou 1])
X_log = torch.linspace(0, 10, 30).unsqueeze(1)
prob_real = 1 / (1 + torch.exp(-(1.5 * (X_log - 5.0))))
Y_log = (prob_real > 0.5).float()

reprovados = X_log[Y_log == 0]
aprovados = X_log[Y_log == 1]
x_curve = np.linspace(0, 10, 100)

# 2. Ajuste Manual Interativo da Sigmoide
def explorar_sigmoide_manual(w=-1.2, b=3.5):
    z = w * X_log.numpy() + b
    pred_prob = 1 / (1 + np.exp(-z))
    
    eps = 1e-7
    pred_prob_clipped = np.clip(pred_prob, eps, 1 - eps)
    bce = -np.mean(Y_log.numpy() * np.log(pred_prob_clipped) + (1 - Y_log.numpy()) * np.log(1 - pred_prob_clipped))
    
    plt.figure(figsize=(9, 5))
    plt.scatter(reprovados.numpy(), np.zeros_like(reprovados.numpy()), color='#FF7043', marker='o', s=70, label='Classe 0: Reprovado', alpha=0.8)
    plt.scatter(aprovados.numpy(), np.ones_like(aprovados.numpy()), color='#7CB342', marker='^', s=70, label='Classe 1: Aprovado', alpha=0.8)
    
    z_c = w * x_curve + b
    sig_c = 1 / (1 + np.exp(-z_c))
    plt.plot(x_curve, sig_c, color='#1BB5D8', linewidth=2.5, label=f'Sigmoide: w={w:.2f}, b={b:.2f}')
    
    plt.axhline(0.5, color='gray', linestyle=':', alpha=0.7, label='Limite de Decisão (P = 0.5)')
    
    if abs(w) > 1e-4:
        x_fronteira = -b / w
        if 0 <= x_fronteira <= 10:
            plt.axvline(x_fronteira, color='#AB47BC', linestyle='--', alpha=0.7, label=f'Fronteira (x={x_fronteira:.2f}h)')
            
    plt.title(f'Ajuste Manual Sigmoide | BCE Loss (Erro): {bce:.4f}', fontsize=12)
    plt.xlabel('Horas de Estudo Semanal de Algoritmos (X)', fontsize=11)
    plt.ylabel('Probabilidade de Aprovação (Y)', fontsize=11)
    plt.legend(fontsize=9, loc='center right')
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.show()

interact(
    explorar_sigmoide_manual,
    w=FloatSlider(min=-3.0, max=4.0, step=0.1, value=-1.2, description='Peso (w):'),
    b=FloatSlider(min=-8.0, max=8.0, step=0.5, value=3.5, description='Viés (b):')
);


In [ ]:
#@title Clique aqui para ver o código { display-mode: "form" }
# 1. Treinando a Regressão Logística com PyTorch (200 Épocas)
w_log = torch.tensor([[-1.2]], requires_grad=True)
b_log = torch.tensor([[3.5]], requires_grad=True)

optimizer_log = optim.SGD([w_log, b_log], lr=0.4)
bce_loss_fn = nn.BCELoss()

historico_log = []

for epoch in range(201):
    z = X_log @ w_log + b_log
    pred_prob = torch.sigmoid(z)
    loss = bce_loss_fn(pred_prob, Y_log)
    
    optimizer_log.zero_grad()
    loss.backward()
    
    g_w = w_log.grad.item()
    g_b = b_log.grad.item()
    grad_norm = np.sqrt(g_w**2 + g_b**2)
    
    historico_log.append({
        'epoch': epoch,
        'w': w_log.item(),
        'b': b_log.item(),
        'loss': loss.item(),
        'grad_norm': grad_norm
    })
    
    optimizer_log.step()

df_hist_log = pd.DataFrame(historico_log)

# 2. Visualização Interativa por Época com ipywidgets
def visualizar_epoca_log(epoca=0):
    dados_ep = df_hist_log.iloc[int(epoca)]
    w_curr, b_curr = dados_ep['w'], dados_ep['b']
    loss_curr, gnorm_curr = dados_ep['loss'], dados_ep['grad_norm']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Subplot 1: Sigmoide se desvirando e criando a fronteira
    ax1.scatter(reprovados.numpy(), np.zeros_like(reprovados.numpy()), color='#FF7043', marker='o', s=50, label='Reprovados (0)', alpha=0.8)
    ax1.scatter(aprovados.numpy(), np.ones_like(aprovados.numpy()), color='#7CB342', marker='^', s=50, label='Aprovados (1)', alpha=0.8)
    ax1.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
    
    z_c = w_curr * x_curve + b_curr
    sig_c = 1 / (1 + np.exp(-z_c))
    ax1.plot(x_curve, sig_c, color='#1BB5D8', linewidth=2.5, 
             label=f'Época {int(epoca)}: w={w_curr:.2f}, b={b_curr:.2f}')
    
    if abs(w_curr) > 1e-4:
        x_front = -b_curr / w_curr
        if 0 <= x_front <= 10:
            ax1.axvline(x_front, color='#AB47BC', linestyle='--', alpha=0.6, label=f'Fronteira (x={x_front:.1f}h)')
            
    ax1.set_title(f'1. Curva Sigmoide Ajustando a Fronteira (Época {int(epoca)})', fontsize=11)
    ax1.set_xlabel('Horas de Estudo (X)')
    ax1.set_ylabel('Probabilidade Predita (Y)')
    ax1.legend(fontsize=9, loc='center right')
    ax1.grid(True, alpha=0.3)
    
    # Subplot 2: Queda da Loss BCE e Diminuição da Norma do Gradiente
    ax2_twin = ax2.twinx()
    
    l1 = ax2.plot(df_hist_log['epoch'][:int(epoca)+1], df_hist_log['loss'][:int(epoca)+1], 
                  color='#0A345D', linewidth=2, label='BCE Loss')
    ax2.scatter(int(epoca), loss_curr, color='#FF7043', s=70, zorder=5)
    
    l2 = ax2_twin.plot(df_hist_log['epoch'][:int(epoca)+1], df_hist_log['grad_norm'][:int(epoca)+1], 
                       color='#7CB342', linestyle='--', linewidth=2, label='Norma do Gradiente ||∇L||')
    ax2_twin.scatter(int(epoca), gnorm_curr, color='#1BB5D8', s=70, zorder=5)
    
    ax2.set_xlim(0, 200)
    ax2.set_ylim(0, df_hist_log['loss'].max() * 1.05)
    ax2_twin.set_ylim(0, df_hist_log['grad_norm'].max() * 1.05)
    
    ax2.set_title('2. Convergência: Queda da Perda BCE e da Norma do Gradiente', fontsize=11)
    ax2.set_xlabel('Época (Iteração)')
    ax2.set_ylabel('BCE Loss', color='#0A345D')
    ax2_twin.set_ylabel('Norma do Gradiente ||∇L||', color='#7CB342')
    
    lines = l1 + l2
    labels = [l.get_label() for l in lines]
    ax2.legend(lines, labels, loc='upper right', fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 [ÉPOCA {int(epoca):03d}]  w = {w_curr:6.3f} | b = {b_curr:6.3f} | BCE Loss = {loss_curr:6.3f} | Norma do Gradiente ||∇L|| = {gnorm_curr:6.3f}")

slider_ep_log = IntSlider(min=0, max=200, step=1, value=0, description='Época:', layout=Layout(width='65%'))
interact(visualizar_epoca_log, epoca=slider_ep_log);


---

## 3. A Ponte Direta para as Redes Neurais Artificiais (MLP)

Agora vem o momento **EUREKA** 💡! Como tudo isso se conecta com os temas da **Aula 02** (Autograd, `nn.Module` e MLPs)?

Regressão Linear, Regressão Logística e Redes Neurais são variações da **mesma ideia fundamental**:

1. **Regressão Linear** = É **1 único neurônio** sem função de ativação!
   $$\hat{y} = w \cdot x + b$$

2. **Regressão Logística** = É **1 único neurônio** com função de ativação **Sigmoide**!
   $$\hat{y} = \sigma(w \cdot x + b)$$

3. **Rede Neural (MLP - Multi-Layer Perceptron)** = É **empilhar vários neurônios** em camadas (Camada Oculta + Camada de Saída), intercalando-os com ativações não-lineares (como **ReLU**)!

```
[Entrada X] ──> [Neurônio 1 (w1*x + b1)] ──> [Ativação ReLU] ──> [Neurônio Saída] ──> Previsão
            ──> [Neurônio 2 (w2*x + b2)] ──> [Ativação ReLU] ──┘
```

Em Deep Learning, **continuamos buscando os coeficientes corretos ($W$ e $b$) da equação**! A única diferença é que em vez de 2 coeficientes ($w$ e $b$), agora temos centenas, milhares ou bilhões de pesos trabalhando em conjunto!


In [ ]:
#@title Clique aqui para ver o código { display-mode: "form" }
# Definindo os 3 modelos no PyTorch para ver a equivalência no código!

# Model 1: Regressão Linear (1 Neurônio Linear)
modelo_linear = nn.Linear(in_features=1, out_features=1)

# Model 2: Regressão Logística (1 Neurônio com Sigmoide)
modelo_logistico = nn.Sequential(
    nn.Linear(in_features=1, out_features=1),
    nn.Sigmoid()
)

# Model 3: Rede Neural MLP (8 Neurônios na Camada Oculta + ReLU + 1 Neurônio Saída com Sigmoide)
modelo_rede_neural = nn.Sequential(
    nn.Linear(in_features=1, out_features=8),   # Camada Oculta (8 neurônios)
    nn.ReLU(),                                   # Ativação Não-Linear
    nn.Linear(in_features=8, out_features=1),   # Camada de Saída (1 neurônio)
    nn.Sigmoid()                                 # Ativação de Saída
)

print("--- ESTRUTURA DOS MODELOS EM PYTORCH ---")
print("1. Regressão Linear:\n", modelo_linear)
print("\n2. Regressão Logística:\n", modelo_logistico)
print("\n3. Rede Neural (MLP):\n", modelo_rede_neural)
